# Spark Silver Processing

This notebook is designed to be run **headlessly** by Airflow via `jupyter nbconvert --execute`.
It reads Bronze data (schedule, matchsheet, lineup) from MinIO and builds Silver Iceberg tables.

**Tables produced:**
- `lake.analytics.teams` (dimension)
- `lake.analytics.players` (dimension)
- `lake.analytics.match_statistics` (fact)
- `lake.analytics.player_match_stats` (fact)

**Do not add extraction or upload cells here.** The DAG handles that separately.

**Gold aggregation (`player_season_stats`) is handled by `spark_gold_processing.ipynb`.**

In [ ]:
# Initialize the Spark session with AQE optimizations
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# The Spark session uses config from spark-defaults.conf (Iceberg + MinIO)
spark = (SparkSession.builder
    .appName("BrasileiraoSilverProcessing")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .getOrCreate())

# Reduce log noise during headless execution
spark.sparkContext.setLogLevel("WARN")
print("Spark session created.")

In [ ]:
import os

# Configuration: Bronze data location in MinIO
RAW_BUCKET = "datalake-raw"

# Season and league are injected by the Airflow DAG:
#   docker exec -e SEASON=<year> -e LEAGUE_KEY=<key>
# Falls back to sensible defaults for local/manual runs.
SEASON = int(os.getenv("SEASON", "2024"))
LEAGUE_KEY = os.getenv("LEAGUE_KEY", "BRA-Brasileirao")

# Map league key → filesystem slug (must match league_config.get_league_slug)
_LEAGUE_SLUGS = {
    "BRA-Brasileirao": "brasileirao",
    "ITA-Serie A": "serie_a",
    "ENG-Premier League": "premier_league",
    "FRA-Ligue 1": "ligue_1",
}
LEAGUE_SLUG = _LEAGUE_SLUGS.get(
    LEAGUE_KEY,
    LEAGUE_KEY.lower().replace(" ", "_").replace("-", "_").strip("_"),
)

BRONZE_URI = f"s3a://{RAW_BUCKET}/espn/{LEAGUE_SLUG}/{SEASON}"

# Read the three Bronze JSONs from MinIO
df_schedule = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/schedule.json")
df_matchsheet = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/matchsheet.json")
df_lineup = spark.read.option("multiline", "true").json(f"{BRONZE_URI}/lineup.json")

print(f"League:     {LEAGUE_KEY} (slug={LEAGUE_SLUG})")
print(f"Season:     {SEASON}")
print(f"Schedule:   {df_schedule.count()} rows")
print(f"Matchsheet: {df_matchsheet.count()} rows")
print(f"Lineup:     {df_lineup.count()} rows")

In [ ]:
# ==========================================================================
# DIMENSION: teams
# Unique team records with venue and capacity from matchsheet data.
# ==========================================================================

# Home teams carry venue/capacity info in matchsheet
df_home_teams = (
    df_matchsheet
    .filter(F.col("is_home") == True)
    .select(
        F.col("team").alias("team_name"),
        F.col("league"),
        F.col("season"),
        F.col("venue"),
        F.col("attendance").cast("int"),
        F.col("capacity").cast("int"),
    )
)

# --- Most common venue (true mode via window, not non-deterministic F.first) ---
df_venue_freq = (
    df_home_teams
    .groupBy("team_name", "league", "season", "venue")
    .agg(F.count("*").alias("venue_count"))
)
venue_window = Window.partitionBy("team_name", "league", "season").orderBy(
    F.desc("venue_count"), "venue"  # secondary sort on name for tie-breaking determinism
)
df_most_common_venue = (
    df_venue_freq
    .withColumn("rn", F.row_number().over(venue_window))
    .filter(F.col("rn") == 1)
    .select("team_name", "league", "season", F.col("venue").alias("home_venue"))
)

# --- Remaining aggregations per team ---
df_team_stats = (
    df_home_teams
    .groupBy("team_name", "league", "season")
    .agg(
        F.max("capacity").cast("int").alias("stadium_capacity"),
        F.round(F.avg("attendance"), 0).cast("int").alias("avg_attendance"),
        F.count("*").alias("home_matches"),
    )
)

# --- Join venue back into team stats ---
df_teams = (
    df_team_stats
    .join(df_most_common_venue, on=["team_name", "league", "season"], how="left")
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Teams dimension: {df_teams.count()} rows")
df_teams.show(truncate=False)

In [ ]:
# ==========================================================================
# DIMENSION: players
# Unique player records derived from lineup data.
# Deduplicated by (player, team), keeping the latest known position.
# ==========================================================================

# Window to pick latest record per player+team (by game date)
player_window = Window.partitionBy("player", "team").orderBy(F.desc("game"))

df_players = (
    df_lineup
    # Rank to deduplicate: keep latest appearance per player+team
    .withColumn("rn", F.row_number().over(player_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        F.col("player"),
        F.col("team"),
        F.col("position"),
        F.col("league"),
        F.col("season"),
    )
)

# Add match count per player+team from the full lineup
df_player_match_count = (
    df_lineup
    .groupBy("player", "team")
    .agg(F.countDistinct("game").alias("matches_played"))
)

# Join match count into the dimension
df_players = (
    df_players
    .join(df_player_match_count, on=["player", "team"], how="left")
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Players dimension: {df_players.count()} rows")
df_players.orderBy(F.desc("matches_played")).show(10, truncate=False)

In [ ]:
# ==========================================================================
# FACT: match_statistics
# Enriched match results joining schedule + derived scores + team stats.
# ==========================================================================

# --- Step 1: Resolve home/away scores ---
# Prefer scores that soccerdata includes directly in the schedule (home_score / away_score).
# Fall back to summing per-player total_goals from lineup only when those columns are missing.
_sched_cols = set(df_schedule.columns)

if "home_score" in _sched_cols and "away_score" in _sched_cols:
    # Schedule already carries scores — use them directly
    df_score_base = df_schedule.select(
        "game",
        F.col("home_score").cast("int").alias("home_score"),
        F.col("away_score").cast("int").alias("away_score"),
    )
    # Remove the raw score columns from the schedule to avoid duplicates on join
    _schedule_for_join = df_schedule.drop("home_score", "away_score")
else:
    # Derive scores by summing total_goals per game + side from the lineup
    df_scores = (
        df_lineup
        .withColumn("total_goals", F.coalesce(F.col("total_goals").cast("int"), F.lit(0)))
        .groupBy("game", "is_home")
        .agg(F.sum("total_goals").alias("team_goals"))
    )
    df_home_sc = df_scores.filter(F.col("is_home") == True).select(
        "game", F.col("team_goals").alias("home_score")
    )
    df_away_sc = df_scores.filter(F.col("is_home") == False).select(
        "game", F.col("team_goals").alias("away_score")
    )
    df_score_base = df_home_sc.join(df_away_sc, on="game", how="outer")
    _schedule_for_join = df_schedule

# --- Step 2: Pivot matchsheet to get home/away stats side by side ---
meta_cols = {"league", "season", "game", "team", "is_home", "venue", "attendance", "capacity"}
stat_cols = [c for c in df_matchsheet.columns if c not in meta_cols]

df_home_stats = df_matchsheet.filter(F.col("is_home") == True)
for col_name in stat_cols:
    df_home_stats = df_home_stats.withColumnRenamed(col_name, f"home_{col_name}")
df_home_stats = df_home_stats.select(
    "game", "venue", "attendance", "capacity",
    *[f"home_{c}" for c in stat_cols]
)

df_away_stats = df_matchsheet.filter(F.col("is_home") == False)
for col_name in stat_cols:
    df_away_stats = df_away_stats.withColumnRenamed(col_name, f"away_{col_name}")
df_away_stats = df_away_stats.select(
    "game",
    *[f"away_{c}" for c in stat_cols]
)

# --- Step 3: Join everything ---
df_match_stats = (
    _schedule_for_join
    .join(df_score_base,   on="game", how="left")
    .join(df_home_stats,   on="game", how="left")
    .join(df_away_stats,   on="game", how="left")
)

# --- Step 4: Computed analytics columns ---
df_match_stats = (
    df_match_stats
    .withColumn("home_score",  F.col("home_score").cast("int"))
    .withColumn("away_score",  F.col("away_score").cast("int"))
    .withColumn("total_goals", F.col("home_score") + F.col("away_score"))
    .withColumn("goal_diff",   F.abs(F.col("home_score") - F.col("away_score")))
    .withColumn("is_draw",     F.col("home_score") == F.col("away_score"))
    .withColumn("winner", F.when(
        F.col("home_score") > F.col("away_score"), F.col("home_team")
    ).when(
        F.col("away_score") > F.col("home_score"), F.col("away_team")
    ).otherwise(F.lit("Draw")))
    .withColumn("processed_at", F.current_timestamp())
)

print(f"Match statistics: {df_match_stats.count()} rows, {len(df_match_stats.columns)} columns")

In [ ]:
# ==========================================================================
# FACT: player_match_stats
# Per-match player performance from lineup data (one row per player per game).
# ==========================================================================

numeric_cols = [
    "total_goals", "goal_assists", "shots_on_target", "total_shots",
    "fouls_committed", "fouls_suffered", "yellow_cards", "red_cards",
    "own_goals", "offsides", "appearances", "saves", "shots_faced",
    "goals_conceded"
]

# Cast numeric columns, filling nulls with 0
df_player_match = df_lineup
for col_name in numeric_cols:
    if col_name in df_player_match.columns:
        df_player_match = df_player_match.withColumn(
            col_name,
            F.coalesce(F.col(col_name).cast("int"), F.lit(0))
        )

# ESPN defines starters. In earlier dumps it might be is_starter.
if "starter" not in df_player_match.columns and "is_starter" in df_player_match.columns:
    df_player_match = df_player_match.withColumn("starter", F.col("is_starter") == True)
elif "starter" not in df_player_match.columns and "formation_place" in df_player_match.columns:
    df_player_match = df_player_match.withColumn("starter", F.col("formation_place").isNotNull())
else:
    # Fallback if the field does not exist at all
    df_player_match = df_player_match.withColumn("starter", F.lit(True))

# Add processing timestamp
df_player_match = df_player_match.withColumn("processed_at", F.current_timestamp())

print(f"Player match stats: {df_player_match.count()} rows")

In [ ]:
# ==========================================================================
# FACT: match_events
# Goals, cards, and substitutions — one row per event per game.
# Source: events.json uploaded by the Bronze DAG (read_events + game_id).
# If the file is absent or empty, this cell is skipped gracefully.
# ==========================================================================

import json as _json

_events_uri = f"{BRONZE_URI}/events.json"

try:
    df_events_raw = spark.read.option("multiline", "true").json(_events_uri)
    _events_available = df_events_raw.count() > 0
except Exception as _exc:
    print(f"events.json not found or unreadable ({_exc}) — skipping match_events table.")
    df_events_raw = None
    _events_available = False

if _events_available:
    # Cast game_id to long if present; ensure season and league columns exist
    df_match_events = df_events_raw
    if "game_id" in df_match_events.columns:
        df_match_events = df_match_events.withColumn("game_id", F.col("game_id").cast("long"))
    if "season" not in df_match_events.columns:
        df_match_events = df_match_events.withColumn("season", F.lit(SEASON).cast("int"))
    # Ensure league column exists for multi-liga partition
    if "league" not in df_match_events.columns:
        df_match_events = df_match_events.withColumn("league", F.lit(LEAGUE_KEY))
    df_match_events = df_match_events.withColumn("processed_at", F.current_timestamp())
    print(f"Match events: {df_match_events.count()} rows, columns: {df_match_events.columns}")
else:
    df_match_events = None
    print("match_events: no data — table will not be written this run.")

In [ ]:
# ==========================================================================
# WRITE: Persist all Silver tables to Iceberg (partitioned by season + league)
# ==========================================================================

spark.sql("CREATE NAMESPACE IF NOT EXISTS lake.analytics")


def _write_partitioned(df, table: str, season: int, league: str) -> None:
    """Idempotent (season, league) partition write.

    First run  → creates the Iceberg table partitioned by (season, league).
    Later runs → atomically overwrites ONLY the matching partition;
                 all other seasons and leagues remain untouched.

    NOTE: If upgrading from single-season partitioning, drop and recreate
    all Silver/Gold tables first using infra/spark/drop_silver_gold_tables.py.
    """
    fq = f"lake.analytics.{table}"
    if spark.catalog.tableExists(fq):
        df.writeTo(fq).overwritePartitions()
        print(f"  {fq}: overwrote (season={season}, league={league}) partition")
    else:
        df.writeTo(fq).partitionedBy("season", "league").create()
        print(f"  {fq}: created (partitioned by season, league)")


_write_partitioned(df_teams,        "teams",              SEASON, LEAGUE_KEY)
_write_partitioned(df_players,      "players",            SEASON, LEAGUE_KEY)
_write_partitioned(df_match_stats,  "match_statistics",   SEASON, LEAGUE_KEY)
_write_partitioned(df_player_match, "player_match_stats", SEASON, LEAGUE_KEY)

# match_events is optional — only write if events.json was populated
if df_match_events is not None:
    _write_partitioned(df_match_events, "match_events", SEASON, LEAGUE_KEY)

print("\nAll Silver Iceberg tables written successfully.")

In [ ]:
# ==========================================================================
# VERIFY: Quick row counts for all tables
# ==========================================================================

tables = [
    "lake.analytics.teams",
    "lake.analytics.players",
    "lake.analytics.match_statistics",
    "lake.analytics.player_match_stats",
    "lake.analytics.match_events",
]

print("Table verification:")
for t in tables:
    try:
        cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {t}").collect()[0]["cnt"]
        print(f"  {t}: {cnt} rows")
    except Exception as exc:
        print(f"  {t}: not found ({exc})")

In [ ]:
# ==========================================================================
# VERIFY: Quick row counts for all tables
# ==========================================================================

tables = [
    "lake.analytics.teams",
    "lake.analytics.players",
    "lake.analytics.match_statistics",
    "lake.analytics.player_match_stats",
]

print("Table verification:")
for t in tables:
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {t}").collect()[0]["cnt"]
    print(f"  {t}: {cnt} rows")

In [ ]:
# Stop the Spark session to free all memory
spark.stop()
print("Spark session stopped. Memory released.")